# 背景污染诊断 — objaverse.xl 多平台版

- 使用 `objaverse.xl` (v2 API) 获取 github/sketchfab/thingiverse/smithsonian 四个平台的模型
- 每个平台抽 5 个 glb 模型，体素化后分组展示
- 打印每个模型的子 mesh 名称和颜色状态

In [ ]:
!pip install objaverse trimesh matplotlib --quiet

In [ ]:
import objaverse.xl as oxl
import numpy as np
import matplotlib.pyplot as plt
import os, shutil, gc, json
from collections import defaultdict
import pandas as pd

SAMPLE_PER_SOURCE = 5
VOXEL_RES = 64
os.makedirs('/content/diag_models', exist_ok=True)
os.makedirs('/content/diag_voxels', exist_ok=True)

print('Loading XL annotations (this downloads parquet files, ~200MB)...')
df = oxl.get_annotations(download_dir='~/.objaverse')
print(f'Total models: {len(df):,}')
print(f'\nSource distribution:')
print(df['source'].value_counts())
print(f'\nFile types:')
print(df['fileType'].value_counts())

In [ ]:
# 每个平台抽 5 个 glb 模型
selected = []
for source in ['sketchfab', 'github', 'thingiverse', 'smithsonian']:
    subset = df[(df['source'] == source) & (df['fileType'] == 'glb')]
    if len(subset) == 0:
        print(f'{source}: 0 glb models, skipping')
        continue
    n = min(SAMPLE_PER_SOURCE, len(subset))
    sampled = subset.sample(n, random_state=42)
    selected.append(sampled)
    print(f'{source:15s}: {len(subset):,} glb models, sampled {n}')

sample_df = pd.concat(selected, ignore_index=True)
print(f'\nTotal to download: {len(sample_df)}')
del df; gc.collect()

In [ ]:
# 下载模型（glb 文件）并保存 fileIdentifier → source 映射
print(f'Downloading {len(sample_df)} models...')
results = oxl.download_objects(objects=sample_df, processes=4)

# 构建映射：本地文件路径 -> (source, fileIdentifier)
path_to_info = {}
for _, row in sample_df.iterrows():
    path_to_info[row['fileIdentifier']] = {
        'source': row['source'],
        'fid': row['fileIdentifier'],
    }

# 下载结果可能是 dict[fileIdentifier, local_path] 或 list
if isinstance(results, dict):
    for fid, local_path in results.items():
        if local_path and os.path.exists(local_path):
            uid = str(abs(hash(fid)))[:16]
            dst = os.path.join('/content/diag_models', f'{uid}.glb')
            shutil.copy(local_path, dst)

            # 保存映射
            with open(f'/content/diag_models/{uid}.json', 'w') as f:
                info = path_to_info.get(fid, {})
                json.dump(info, f)
else:
    print(f'Unexpected results type: {type(results)}')
    print(f'Results: {list(results)[:3] if results else "empty"}')

model_files = [f for f in os.listdir('/content/diag_models') if f.endswith('.glb')]
print(f'Downloaded {len(model_files)} models (with metadata)')


In [ ]:
# 体素化 + 颜色采样
import trimesh

def sample_colors(mesh, points, face_idx):
    # 路径 1: PBR 贴图 baseColorTexture
    try:
        if (hasattr(mesh.visual, 'material')
                and hasattr(mesh.visual.material, 'baseColorTexture')
                and mesh.visual.material.baseColorTexture is not None):
            texture = np.array(mesh.visual.material.baseColorTexture)
            uv = mesh.visual.uv
            if uv is not None and len(uv) > 0:
                face_uv = uv[mesh.faces[face_idx]]
                bary = trimesh.triangles.points_to_barycentric(
                    mesh.triangles[face_idx], points)
                sample_uv = (face_uv * bary[:, :, None]).sum(axis=1)
                H, W = texture.shape[:2]
                u, v = sample_uv[:, 0], 1.0 - sample_uv[:, 1]
                px = np.clip((u * (W - 1)).astype(int), 0, W - 1)
                py = np.clip((v * (H - 1)).astype(int), 0, H - 1)
                return texture[py, px][:, :3]
    except Exception:
        pass

    # 路径 2: PBR 纯色 baseColorFactor
    try:
        if (hasattr(mesh.visual, 'material')
                and hasattr(mesh.visual.material, 'baseColorFactor')):
            c = np.array(mesh.visual.material.baseColorFactor[:3]).astype(float)
            if c.max() <= 1.0:
                c = c * 255  # glTF spec: [0,1] -> [0,255]
            return np.tile(c.clip(0, 255).astype(np.uint8), (len(points), 1))
    except Exception:
        pass

    # 路径 3: vertex_colors
    try:
        if hasattr(mesh.visual, 'vertex_colors') and mesh.visual.vertex_colors is not None:
            vc = mesh.visual.vertex_colors
            if vc.ndim == 2 and vc.shape[1] >= 3:
                from trimesh.visual.color import vertex_to_face_color
                fc = vertex_to_face_color(vc, mesh.faces)
                return fc[face_idx][:, :3]
    except Exception:
        pass

    # 路径 4: PBR 材质存在但未指定颜色, glTF 默认白色
    try:
        if hasattr(mesh.visual, 'material'):
            return np.tile(np.array([255, 255, 255], dtype=np.uint8), (len(points), 1))
    except Exception:
        pass

    # 路径 5: 灰色默认
    return np.tile([128, 128, 128], (len(points), 1))


def voxelize(file_path, resolution=64):
    try:
        scene = trimesh.load(file_path)
        if isinstance(scene, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(scene.geometry.values()))
        else:
            mesh = scene
        pts, fidx = trimesh.sample.sample_surface(mesh, 200000)
        colors = sample_colors(mesh, pts, fidx)
        mins, maxs = pts.min(axis=0), pts.max(axis=0)
        pts_n = (pts - mins) / (maxs - mins + 1e-8)
        coords = np.clip((pts_n * (resolution - 1)).astype(int), 0, resolution - 1)
        vox = np.zeros((resolution, resolution, resolution, 4), dtype=np.uint8)
        for c, col in zip(coords, colors):
            vox[c[0], c[1], c[2], :3] = col
            vox[c[0], c[1], c[2], 3] = 255
        return vox
    except:
        return None


# 体素化所有模型
results = {}
model_dir = '/content/diag_models'
for fname in sorted(os.listdir(model_dir)):
    if not fname.endswith('.glb'):
        continue
    uid = fname.replace('.glb', '')
    fpath = os.path.join(model_dir, fname)

    # 读取该模型的元数据 (source, fileIdentifier)
    meta_path = os.path.join(model_dir, f'{uid}.json')
    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            info = json.load(f)
        source = info.get('source', 'unknown')
        model_url = info.get('fid', uid)[:60]
    else:
        source = 'unknown'
        model_url = uid[:20]

    # 统计子 mesh
    submesh_names = []
    try:
        scene = trimesh.load(fpath, force='scene')
        if isinstance(scene, trimesh.Scene):
            for i, g in enumerate(scene.geometry.values()):
                nm = g.metadata.get('name', f'mesh_{i}') if hasattr(g, 'metadata') else f'mesh_{i}'
                submesh_names.append(nm)
            subm_count = len(submesh_names)
        else:
            subm_count = 1
            submesh_names = ['main']
    except:
        subm_count = '?'
        submesh_names = ['?']

    vox = voxelize(fpath, VOXEL_RES)

    # 质量检查: 检测体素是否被背景平面污染
    skip_model = False
    quality_note = ''
    if vox is not None:
        fill_ratio = (vox[..., 3] > 0).mean()
        # 填充率 > 40% 通常意味着有大面积背景平面
        if fill_ratio > 0.40:
            skip_model = True
            quality_note = f'SKIPPED: fill={fill_ratio:.1%} (likely background plane)'
        elif fill_ratio < 0.003:
            skip_model = True
            quality_note = f'SKIPPED: fill={fill_ratio:.1%} (nearly empty)'

    if vox is not None:
        has_color = np.any(vox[..., :3].sum(axis=-1) != 128 * 3 * (vox[..., 3] > 0))
    if skip_model:
        print(f'  [{source:<12s}] {model_url[:60]:<62s} | {quality_note}')
        continue

        results[uid] = {
            'voxel': vox, 'source': source,
            'submesh_count': subm_count, 'submesh_names': submesh_names,
            'uid': uid, 'model_url': model_url,
        }
        print(f'  [{source:<12s}] {model_url[:60]:<62s} | {subm_count} parts | {"COLOR" if has_color else "GRAY"}')
    else:
        print(f'  [{source:<12s}] {model_url[:60]:<62s} | FAILED')

print(f'\nVoxelized {len(results)} models')


In [ ]:
# 按平台分组可视化
grouped = defaultdict(list)
for uid, info in results.items():
    grouped[info['source']].append(info)

print(f'DEBUG: {len(results)} models, sources: {dict((k, len(v)) for k, v in grouped.items())}\n')

for source in sorted(grouped.keys()):
    items = grouped[source]
    n = len(items)
    cols = min(n, 5)
    rows = max(1, (n + cols - 1) // cols)

    fig = plt.figure(figsize=(4 * cols, 4 * rows))
    fig.suptitle(f'{source.upper()} ({n} models)', fontsize=14, fontweight='bold', y=0.98)

    for i, item in enumerate(items):
        voxel = item['voxel']
        occ = voxel[..., 3] > 0
        colors = voxel[..., :3].astype(float) / 255.0

        try:
            ax = fig.add_subplot(rows, cols, i + 1, projection='3d')
            ax.voxels(occ, facecolors=colors, edgecolor=None)
            fill_pct = occ.mean() * 100
            subm = item['submesh_count']
            ax.set_title(f'{item["uid"][:8]}... | {subm} sub | {fill_pct:.1f}%', fontsize=8)
            ax.axis('off')
        except Exception as e:
            print(f'  Render error for {item["uid"][:12]}: {e}')

    plt.tight_layout()
    path = f'/content/diag_voxels/{source}_samples.png'
    plt.savefig(path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {path}\n')

In [ ]:
# 手动深入检查某个模型的所有子 mesh 材质
# 把 INSPECT_UID 改成你想分析的 uid，重新运行此 cell

INSPECT_UID = None

if INSPECT_UID and any(INSPECT_UID in k for k in results):
    fname = None
    for f in os.listdir('/content/diag_models'):
        if INSPECT_UID in f:
            fname = f
            break

    if fname:
        import trimesh
        scene = trimesh.load(os.path.join('/content/diag_models', fname), force='scene')
        print(f'Model: {fname}')
        print(f'Scene type: {type(scene).__name__}')

        if isinstance(scene, trimesh.Scene):
            for i, (name, geom) in enumerate(scene.geometry.items()):
                print(f'\n--- Submesh {i}: {name} ---')
                print(f'  Vertices: {len(geom.vertices)}, Faces: {len(geom.faces)}')
                print(f'  Visual type: {type(geom.visual).__name__}')

                if hasattr(geom.visual, 'material'):
                    mat = geom.visual.material
                    print(f'  Material type: {type(mat).__name__}')
                    for attr in ['baseColorFactor', 'baseColorTexture', 'metallicFactor', 'roughnessFactor']:
                        if hasattr(mat, attr):
                            val = getattr(mat, attr)
                            if val is not None:
                                val_str = str(val)[:100]
                                print(f'    {attr}: {val_str}')
                else:
                    print(f'  No material')

                if hasattr(geom.visual, 'vertex_colors'):
                    vc = geom.visual.vertex_colors
                    print(f'  vertex_colors: shape={vc.shape if vc is not None else None}')

                if hasattr(geom.visual, 'uv'):
                    uv = geom.visual.uv
                    print(f'  UV: shape={uv.shape if uv is not None else None}')
else:
    print('修改 INSPECT_UID 变量为某个 uid 前缀，重新运行此 cell')
    print(f'Available UIDs: {list(results.keys())[:5]}')

In [ ]:
# 查看每个模型的 fileIdentifier URL（包含模型名称和来源页面链接）
import json, os

model_dir = '/content/diag_models'
print(f'{"#":<4s} {"Source":<14s} {"Model URL / fileIdentifier"}')
print('-' * 100)

for i, fname in enumerate(sorted(os.listdir(model_dir))):
    if not fname.endswith('.json'):
        continue
    with open(os.path.join(model_dir, fname), 'r') as f:
        info = json.load(f)
    source = info.get('source', '?')
    fid = info.get('fid', '')

    # 尝试从 URL 提取模型名
    model_name = fid
    if 'sketchfab.com/3d-models/' in fid:
        parts = fid.split('/3d-models/')
        if len(parts) > 1:
            model_name = f'sketchfab: {parts[1][:50]}'
    elif 'github.com/' in fid:
        parts = fid.split('github.com/')
        if len(parts) > 1:
            model_name = f'github: {parts[1][:50]}'
    elif 'thingiverse.com/' in fid:
        parts = fid.split('thingiverse.com/')
        if len(parts) > 1:
            model_name = f'thingiverse: {parts[1][:50]}'
    elif '3d.si.edu' in fid:
        model_name = f'smithsonian: {fid[-50:]}'

    print(f'{i:<4d} [{source:<12s}] {model_name[:100]}')
    print(f'      {" " * (len(source)+2)}URL: {fid[:120]}')
    print()

print('\n提示：复制 Sketchfab URL 在浏览器打开可以看到模型标题和描述。')


In [ ]:
# 查看指定模型的完整信息
# 把 MODEL_TO_CHECK 改成上面打印出的某个 URL 片段

MODEL_TO_CHECK = ''  # 例如: '606b9f3ddd' (uid前缀) 或 'sketchfab: xx' (模型名片段)

import json, os
model_dir = '/content/diag_models'

for fname in sorted(os.listdir(model_dir)):
    if not fname.endswith('.json'):
        continue
    with open(os.path.join(model_dir, fname), 'r') as f:
        info = json.load(f)
    uid = fname.replace('.json', '')
    fid = info.get('fid', '')

    if MODEL_TO_CHECK and MODEL_TO_CHECK not in uid and MODEL_TO_CHECK not in fid:
        continue

    print(f'UID: {uid}')
    print(f'Source: {info.get("source", "?")}')
    print(f'FileIdentifier: {fid}')
    print()
    print('glb file:', os.path.join(model_dir, f'{uid}.glb'))
    print('json file:', os.path.join(model_dir, f'{uid}.json'))

    # 如果之前跑过材质分析，results 字典里会有详细信息
    if 'results' in dir() and uid in results:
        r = results[uid]
        print(f'Sub-meshes: {r["submesh_count"]}')
        print(f'Names: {r.get("submesh_names", [])[:10]}')

    print('-' * 60)

if not MODEL_TO_CHECK:
    print('设置 MODEL_TO_CHECK 为 uid 前缀或 URL 片段，重新运行。')
